# LDA Topic Analysis and Polarisation Workflow

This notebook is organized for reruns and hand-in: shared path setup first, then the main LDA and scoring pipeline, then archived code at the end.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

from pathlib import Path
from huggingface_hub import list_repo_files
from datasets import load_dataset
from openai import OpenAI
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from rapidfuzz import process, fuzz

import json
import os
import re
import time
import unicodedata
import numpy as np
import pandas as pd
import requests
import subprocess, sys

# ------------------------------
# Reusable project path setup
# ------------------------------
USERNAME = "gebruiker"
PROJECT_FOLDER_NAME = "MEP"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if p.name == PROJECT_FOLDER_NAME),
    NOTEBOOK_DIR.parent,
)

LDA_DIR = PROJECT_ROOT / "4 LDA Topic Analysis"
WIKIPEDIA_DIR = PROJECT_ROOT / "4 Wikipedia"
LLM_WIKIPEDIA_DIR = PROJECT_ROOT / "5 LLM Wikipedia analysis"

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = LDA_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Candidate paths for inputs that may exist in different versions of the project structure.
CLUSTER_SUMMARY_CANDIDATES = [
    TABLES_DIR / "mep_cluster_summary.xlsx",
    LDA_DIR / "mep_cluster_summary.xlsx",
]
WIKIPEDIA_CANDIDATES = [
    WIKIPEDIA_DIR / "outputs" / "tables" / "mep_wikipedia_final.xlsx",
    WIKIPEDIA_DIR / "mep_wikipedia_final.xlsx",
]
EXTRACTED_POLARISATION_CANDIDATES = [
    LLM_WIKIPEDIA_DIR / "extracted_polarisation_final.xlsx",
    LLM_WIKIPEDIA_DIR / "outputs" / "tables" / "extracted_polarisation_final.xlsx",
]

CLUSTER_SUMMARY_PATH = next((p for p in CLUSTER_SUMMARY_CANDIDATES if p.exists()), CLUSTER_SUMMARY_CANDIDATES[0])
WIKIPEDIA_PATH = next((p for p in WIKIPEDIA_CANDIDATES if p.exists()), WIKIPEDIA_CANDIDATES[0])
EXTRACTED_POLARISATION_PATH = next((p for p in EXTRACTED_POLARISATION_CANDIDATES if p.exists()), EXTRACTED_POLARISATION_CANDIDATES[0])

print(f"Project root: {PROJECT_ROOT}")
print(f"Tables dir:   {TABLES_DIR}")
print(f"Figures dir:  {FIGURES_DIR}")
print(f"Cluster file: {CLUSTER_SUMMARY_PATH}")
print(f"Wiki file:    {WIKIPEDIA_PATH}")



## Optional dataset availability check

This quick check only lists available files from the remote dataset repository.
It is optional and does not affect the core pipeline.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [8]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# Optional quick check (safe to skip if repo access is restricted)

try:
    files = list_repo_files("coastalcph/eu_debates")
    print([f for f in files if f.endswith((".parquet", ".jsonl", ".zip"))])
except Exception as e:
    print("Repo file listing skipped:", e)
    print("Proceed with load_dataset('RJuro/eu_debates') in the next cells.")


Repo file listing skipped: 401 Client Error. (Request ID: Root=1-6a34085e-28cda75b13b30a131abb820d;5c132d7c-a376-481e-ae3c-13e16cb27853)

Repository Not Found for url: https://huggingface.co/api/models/coastalcph/eu_debates/tree/main?recursive=true&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Proceed with load_dataset('RJuro/eu_debates') in the next cells.


## Load MEP Speeches and Build Topic Pipeline

This section now follows the updated workflow:
1. Load all MEP speeches from `RJuro/eu_debates`.
2. Build MEP-year speech documents.
3. Filter to demography-linked MEP-year documents via vote activity.
4. Train LDA and compute document/MEP-level topic shares.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.


# ------------------------------
# Global settings and time range
# ------------------------------
BASE = "https://howtheyvote.eu"
START_DATE = pd.Timestamp("2019-07-02")
END_DATE = pd.Timestamp("2024-07-16")

# ------------------------------
# Name normalization helper
# Makes joins more robust across sources with different punctuation/accents
# ------------------------------
def canonical_name(name: str) -> str:
    if pd.isna(name):
        return pd.NA
    name = str(name)
    name = "".join(
        c for c in unicodedata.normalize("NFKD", name)
        if not unicodedata.combining(c)
    )
    name = name.lower()
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name if name else pd.NA

# ------------------------------
# Vote search helper (HowTheyVote API)
# Returns vote metadata for a query in the selected time window
# ------------------------------
def fetch_votes(query: str, page_size: int = 100, max_pages: int | None = None) -> pd.DataFrame:
    fields = [
        "id", "timestamp", "display_title", "reference", "result",
    ]
    all_results = []
    page = 1

    while True:
        r = requests.get(
            f"{BASE}/api/votes/search",
            params={"q": query, "page": page, "page_size": page_size},
            timeout=60,
        )
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])
        if not results:
            break

        all_results.extend(results)
        if not data.get("next"):
            break

        page += 1
        if max_pages is not None and page > max_pages:
            break

    df_votes = pd.DataFrame([{k: v.get(k) for k in fields} for v in all_results])
    if df_votes.empty:
        return pd.DataFrame(columns=["vote_id", "timestamp", "date", "year", "title", "reference", "result"] )

    df_votes = df_votes.rename(columns={"id": "vote_id", "display_title": "title"})
    df_votes["timestamp"] = pd.to_datetime(df_votes["timestamp"], errors="coerce")
    df_votes["date"] = df_votes["timestamp"].dt.date
    df_votes["year"] = df_votes["timestamp"].dt.year

    # Restrict to 9th term window used in main workflow
    mask = (df_votes["timestamp"] >= START_DATE) & (df_votes["timestamp"] <= END_DATE)
    return df_votes[mask].reset_index(drop=True)

# ------------------------------
# Download full roll-call for a specific vote
# ------------------------------
def load_vote_details(vote_id: int) -> pd.DataFrame:
    url = f"{BASE}/api/votes/{int(vote_id)}.csv"
    return pd.read_csv(url)


### Build Speech Documents

This next cell loads all available MEP speeches and aggregates them into one document per MEP-year.
Output table: `speech_docs_per_mep_year`.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [10]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ==============================================================
# STEP 1: Load all MEP speeches from the EU debates dataset
# ==============================================================
eu = load_dataset("RJuro/eu_debates", split="train")
debates_df = eu.to_pandas()

# Prefer translated text if available, otherwise use original text.
# This matches the logic in your main notebook.
debates_df["text_final"] = debates_df["translated_text"].fillna(debates_df["text"] )
debates_df = debates_df.dropna(subset=["text_final"]).copy()

# Build normalized name key for cross-source matching
debates_df["mep_name_raw"] = debates_df["speaker_name"]
debates_df["mep_name_clean"] = debates_df["mep_name_raw"].apply(canonical_name)

# ==============================================================
# STEP 2: Aggregate to one document per MEP-year
# ==============================================================
speech_docs_per_mep_year = (
    debates_df
    .dropna(subset=["mep_name_clean", "year"])
    .groupby(["mep_name_clean", "year"], as_index=False)
    .agg(
        # Concatenate all speeches in same MEP-year into one analysis document
        text=("text_final", lambda x: "\n\n".join(x.dropna().astype(str))),
        mep_name_raw=("mep_name_raw", "first"),
        # Use most frequent party label in that MEP-year
        speaker_party=("speaker_party", lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA),
    )
)

# Add channel and stable document id
speech_docs_per_mep_year["channel"] = "speech"
speech_docs_per_mep_year["doc_id"] = (
    "speech_"
    + speech_docs_per_mep_year["mep_name_clean"].astype(str)
    + "_y"
    + speech_docs_per_mep_year["year"].astype(str)
)

speech_docs_per_mep_year = speech_docs_per_mep_year[[
    "doc_id", "channel", "mep_name_clean", "mep_name_raw", "year", "speaker_party", "text"
]]

print("All speech documents:", speech_docs_per_mep_year.shape)
speech_docs_per_mep_year.head()


All speech documents: (10912, 7)


,doc_id,channel,mep_name_clean,mep_name_raw,year,speaker_party,text
0,speech_abdullah ii_y2012,speech,abdullah ii,Abdullah II,2012,N/A,"Mr President, distinguished Members of the Eur..."
1,speech_abdullah ii_y2020,speech,abdullah ii,Abdullah II,2020,N/A,King of the Hashemite Kingdom of Jordan. Ã¢â‚¬â€œ Bi...
2,speech_abir al sahlani_y2019,speech,abir al sahlani,Abir Al-Sahlani,2019,ALDE,"The writer. Ã¢â‚¬â€œ Mr President, dear colleagues! I..."
3,speech_abir al sahlani_y2020,speech,abir al sahlani,Abir Al-Sahlani,2020,ALDE,The Renew Group. The President! Authority regi...
4,speech_abir al sahlani_y2021,speech,abir al sahlani,Abir Al-Sahlani,2021,ALDE,"The President! Thank you, Minister and Secret..."


## Demography Topic Pipeline

This section extends the workflow in three steps:
1. Filter MEP-year speeches using demography-related votes (instead of only immigration).
2. Train an LDA model with top X topics and assign topic-probability vectors to each speech document.
3. Aggregate average topic shares per MEP.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ==============================================================
# STEP 6: Build a stricter demography-focused document set
# Logic:
# 1) Keep only speeches that are directly demography-relevant.
# 2) Do NOT intersect with vote activity yet.
# 3) Aggregate matched speeches to MEP-year level.
# ==============================================================

# ------------------------------
# A) Core demographic keywords
# ------------------------------

CORE_DEMOGRAPHY_KEYWORDS = [
    # General demography
    "demograph",
    "population ageing",
    "population aging",
    "population decline",
    "population growth",
    "population structure",
    "age structure",
    "dependency ratio",

    # Fertility and births
    "fertility",
    "birth rate",
    "birth rates",
    "natality",
    "replacement rate",

    # Ageing
    "ageing population",
    "aging population",
    "older population",
    "elderly population",
    "working-age population",

    # Depopulation
    "shrinking population",
    "rural depopulation",
    "depopulation",
    "demographic decline",

    # Labour-market demography
    "labour shortage",
    "labour shortages",
    "labor shortage",
    "labor shortages",
    "workforce shortage",
    "workforce shortages",

    # Care and pensions, only when quite specific
    "long-term care",
    "pension system",
    "pension systems",
    "retirement age",

    # Migration as demographic balance, not migration generally
    "migration balance",
    "net migration",
]

CORE_DEMOGRAPHY_REGEX = re.compile(
    "|".join(re.escape(k) for k in CORE_DEMOGRAPHY_KEYWORDS),
    flags=re.IGNORECASE
)

# ------------------------------
# B) Speech-level demography relevance
# ------------------------------

debates_demography = debates_df.copy()

debates_demography["_text_for_filter"] = (
    debates_demography["text_final"].fillna("").astype(str)
    + "\n"
    + debates_demography["debate_title"].fillna("").astype(str)
)

debates_demography = debates_demography[
    debates_demography["_text_for_filter"].str.contains(
        CORE_DEMOGRAPHY_REGEX,
        regex=True,
        na=False
    )
].copy()

print("Matched demography speeches:", debates_demography.shape)

# ------------------------------
# C) Aggregate to MEP-year documents
# ------------------------------

docs_demography = (
    debates_demography
    .dropna(subset=["mep_name_clean", "year"])
    .groupby(["mep_name_clean", "year"], as_index=False)
    .agg(
        text=("text_final", lambda x: "\n\n".join(x.dropna().astype(str))),
        mep_name_raw=("mep_name_raw", "first"),
        speaker_party=("speaker_party", lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA),
        n_demography_speeches=("text_final", "size"),
    )
)

docs_demography["channel"] = "speech"

docs_demography["doc_id"] = (
    "speech_demo_"
    + docs_demography["mep_name_clean"].astype(str)
    + "_y"
    + docs_demography["year"].astype(str)
)

docs_demography = docs_demography[[
    "doc_id",
    "channel",
    "mep_name_clean",
    "mep_name_raw",
    "year",
    "speaker_party",
    "text",
    "n_demography_speeches",
]]

docs_demography = docs_demography.reset_index(drop=True)

print("Demography speech docs for topic modelling:", docs_demography.shape)
docs_demography.head()


Matched demography speeches: (1337, 14)
Demography speech docs for topic modelling: (975, 8)


,doc_id,channel,mep_name_clean,mep_name_raw,year,speaker_party,text,n_demography_speeches
0,speech_demo_abdullah ii_y2012,speech,abdullah ii,Abdullah II,2012,N/A,"Mr President, distinguished Members of the Eur...",1
1,speech_demo_abir al sahlani_y2021,speech,abir al sahlani,Abir Al-Sahlani,2021,N/A,"rapporteur. Ã¢â‚¬â€œ MadamÃ‚Â President, we are 27 Membe...",1
2,speech_demo_abir al sahlani_y2022,speech,abir al sahlani,Abir Al-Sahlani,2022,ALDE,"on behalf of the Renew Group. Ã¢â‚¬â€œ Mr President, ...",1
3,speech_demo_adam gierek_y2019,speech,adam gierek,Adam Gierek,2019,S&D,Less ambition and more efficiency. I think the...,1
4,speech_demo_adam kosa_y2020,speech,adam kosa,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,2020,PPE,The President Mrs. The epidemic more than any...,1


In [24]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ==============================================================
# STEP 7: LDA on demography-filtered speech documents
# ==============================================================

# Increase this to allow a broader topic space
TOPIC_COUNT = 50

# Keep only non-empty texts
docs_demography_lda = docs_demography.copy()
docs_demography_lda["text"] = docs_demography_lda["text"].fillna("").astype(str)
docs_demography_lda = docs_demography_lda[docs_demography_lda["text"].str.strip() != ""].copy()

if docs_demography_lda.empty:
    raise ValueError("No text documents available after demography filtering.")

# BoW features for LDA
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=5,
    max_df=0.7,
    max_features=5000,
    ngram_range=(1, 2),
)

X_bow = vectorizer.fit_transform(docs_demography_lda["text"] )

lda_model = LatentDirichletAllocation(
    n_components=TOPIC_COUNT,
    random_state=42,
    learning_method="batch",
    max_iter=30,
)

doc_topic_matrix = lda_model.fit_transform(X_bow)

# Add topic probability vector columns to each document
topic_cols = [f"topic_{i:02d}" for i in range(TOPIC_COUNT)]
docs_demography_lda[topic_cols] = doc_topic_matrix
docs_demography_lda["dominant_topic"] = docs_demography_lda[topic_cols].idxmax(axis=1)

# Show top terms per topic for interpretation
feature_names = np.array(vectorizer.get_feature_names_out())
TOP_TERMS_PER_TOPIC = 12

topic_rows = []
for i, comp in enumerate(lda_model.components_):
    top_idx = np.argsort(comp)[::-1][:TOP_TERMS_PER_TOPIC]
    topic_rows.append({
        "topic": f"topic_{i:02d}",
        "top_terms": ", ".join(feature_names[top_idx]),
    })

topic_terms_df = pd.DataFrame(topic_rows)

print("LDA documents:", docs_demography_lda.shape)
print("Topic matrix shape:", doc_topic_matrix.shape)
display(topic_terms_df)
docs_demography_lda.head()


LDA documents: (975, 59)
Topic matrix shape: (975, 50)


,topic,top_terms
0,topic_00,"pension, pensions, pension systems, systems, m..."
1,topic_01,"water, population, population growth, climate,..."
2,topic_02,"union, european union, think, dear, colleagues..."
3,topic_03,"women, health, rights, gender, equality, membe..."
4,topic_04,"union, european union, europe, economic, hunga..."
5,topic_05,"europe, union, ukraine, world, need, work, peo..."
6,topic_06,"council, parliament, european council, treaty,..."
7,topic_07,"europe, solidarity, countries, president, unio..."
8,topic_08,"president, mr, union, european union, mr presi..."
9,topic_09,"president, women, mr, mr president, parliament..."


,doc_id,channel,mep_name_clean,mep_name_raw,year,speaker_party,text,n_demography_speeches,topic_00,topic_01,...,topic_41,topic_42,topic_43,topic_44,topic_45,topic_46,topic_47,topic_48,topic_49,dominant_topic
0,speech_demo_abdullah ii_y2012,speech,abdullah ii,Abdullah II,2012,N/A,"Mr President, distinguished Members of the Eur...",1,0.000033,0.000033,...,0.000033,0.000033,0.000033,0.000033,0.000033,0.000033,0.000033,0.000033,0.000033,topic_39
1,speech_demo_abir al sahlani_y2021,speech,abir al sahlani,Abir Al-Sahlani,2021,N/A,"rapporteur. Ã¢â‚¬â€œ MadamÃ‚Â President, we are 27 Membe...",1,0.000054,0.000054,...,0.004623,0.000054,0.000054,0.000054,0.000054,0.000054,0.000054,0.000054,0.000054,topic_37
2,speech_demo_abir al sahlani_y2022,speech,abir al sahlani,Abir Al-Sahlani,2022,ALDE,"on behalf of the Renew Group. Ã¢â‚¬â€œ Mr President, ...",1,0.000110,0.000110,...,0.000110,0.000110,0.000110,0.000110,0.000110,0.000110,0.000110,0.000110,0.000110,topic_37
3,speech_demo_adam gierek_y2019,speech,adam gierek,Adam Gierek,2019,S&D,Less ambition and more efficiency. I think the...,1,0.000227,0.000227,...,0.000227,0.000227,0.000227,0.000227,0.000227,0.000227,0.000227,0.000227,0.000227,topic_34
4,speech_demo_adam kosa_y2020,speech,adam kosa,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,2020,PPE,The President Mrs. The epidemic more than any...,1,0.000263,0.000263,...,0.000263,0.000263,0.000263,0.000263,0.000263,0.000263,0.000263,0.000263,0.000263,topic_08


## Topic Cluster Assignments

Topics are grouped into substantive clusters based on their top keywords.
Topics not explicitly assigned fall into the **Other** cluster.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [26]:
# Purpose: Run the core processing logic for this stage of the notebook workflow.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ==============================================================
# Topic Cluster Assignments
# Maps each topic to the substantive cluster you defined.
# Topics not listed below default to 'Other'.
# ==============================================================

# Cluster mapping: cluster name -> list of topic indices (integers)
CLUSTER_MAP = {
    "Ageing & Pensions":              [0, 17, 24, 28, 45],
    "Fertility & Family Formation":   [3, 9, 18],
    "Migration & Population Change":  [34],
    "Youth & Future Generations":     [2, 64, 68],
    "Labour Force & Workforce":       [31, 48],
    "Regional Depopulation":          [47, 60],
    "Health & Care Services":         [43],
    "Housing & Living Conditions":    [44],
    "Population Growth & Sustainability": [1, 4],
}

# Build a reverse lookup: topic_index -> cluster name
topic_idx_to_cluster = {}
for cluster_name, indices in CLUSTER_MAP.items():
    for idx in indices:
        topic_idx_to_cluster[idx] = cluster_name

# Apply to topic_terms_df
topic_clustered_df = topic_terms_df.copy()
topic_clustered_df["topic_index"] = topic_clustered_df["topic"].str.extract(r"(\d+)").astype(int)
topic_clustered_df["cluster"] = (
    topic_clustered_df["topic_index"]
    .map(topic_idx_to_cluster)
    .fillna("Other")
)

# Keep only topics within the current TOPIC_COUNT range
topic_clustered_df = topic_clustered_df[
    topic_clustered_df["topic_index"] < TOPIC_COUNT
].copy()

# Display sorted by cluster
display(
    topic_clustered_df[["topic", "cluster", "top_terms"]]
    .sort_values(["cluster", "topic"])
    .reset_index(drop=True)
)


,topic,cluster,top_terms
0,topic_00,Ageing & Pensions,"pension, pensions, pension systems, systems, m..."
1,topic_17,Ageing & Pensions,"asylum, migration, seekers, asylum seekers, ro..."
2,topic_24,Ageing & Pensions,"pension, people, pepp, systems, product, publi..."
3,topic_28,Ageing & Pensions,"life, council, europeans, migration, socialist..."
4,topic_45,Ageing & Pensions,"ireland, island, eu, european union, citizens,..."
5,topic_03,Fertility & Family Formation,"women, health, rights, gender, equality, membe..."
6,topic_09,Fertility & Family Formation,"president, women, mr, mr president, parliament..."
7,topic_18,Fertility & Family Formation,"member, states, member states, development, po..."
8,topic_43,Health & Care Services,"commission, social, member, health, care, supp..."
9,topic_44,Housing & Living Conditions,"housing, people, report, think, europe, eu, pr..."


## MEP Average Topic Shares by Cluster

For each MEP, this section computes the average share of speech devoted to each cluster,
by summing the topic probabilities of all topics belonging to that cluster.
The result is one row per MEP with one column per cluster.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [27]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ==============================================================
# MEP average cluster shares
# Requires: docs_demography_lda, topic_cols, CLUSTER_MAP, TOPIC_COUNT
# ==============================================================

# 1) Build cluster -> topic column list mapping
#    Only include topic indices that exist in this run
cluster_to_topic_cols = {}
for cluster_name, indices in CLUSTER_MAP.items():
    cols = [f"topic_{i:02d}" for i in indices if i < TOPIC_COUNT]
    if cols:
        cluster_to_topic_cols[cluster_name] = cols

# 2) Sum probabilities of all topics in each cluster for every document
docs_with_clusters = docs_demography_lda[["doc_id", "mep_name_clean", "mep_name_raw"] + topic_cols].copy()

all_cluster_names = list(cluster_to_topic_cols.keys()) + ["Other"]

# Named clusters: sum their topic probabilities per document
for cluster_name, cols in cluster_to_topic_cols.items():
    docs_with_clusters[cluster_name] = docs_with_clusters[cols].sum(axis=1)

# 'Other' cluster: any probability not assigned to a named cluster
named_cols = [c for cols in cluster_to_topic_cols.values() for c in cols]
other_cols = [c for c in topic_cols if c not in named_cols]
if other_cols:
    docs_with_clusters["Other"] = docs_with_clusters[other_cols].sum(axis=1)
else:
    docs_with_clusters["Other"] = 0.0

cluster_cols = list(cluster_to_topic_cols.keys()) + ["Other"]

# 3) Average cluster shares per MEP across all their documents
mep_cluster_avg = (
    docs_with_clusters
    .groupby(["mep_name_clean", "mep_name_raw"], as_index=False)[cluster_cols]
    .mean()
)

# 4) Add document count per MEP
mep_n_docs = (
    docs_demography_lda
    .groupby(["mep_name_clean", "mep_name_raw"], as_index=False)
    .agg(n_documents=("doc_id", "size"))
)

mep_cluster_summary = mep_n_docs.merge(
    mep_cluster_avg,
    on=["mep_name_clean", "mep_name_raw"],
    how="left",
)

# 5) Dominant cluster per MEP
mep_cluster_summary["dominant_cluster"] = mep_cluster_summary[cluster_cols].idxmax(axis=1)

print("MEP cluster summary shape:", mep_cluster_summary.shape)
mep_cluster_summary.head(20)


MEP cluster summary shape: (668, 14)


,mep_name_clean,mep_name_raw,n_documents,Ageing & Pensions,Fertility & Family Formation,Migration & Population Change,Youth & Future Generations,Labour Force & Workforce,Regional Depopulation,Health & Care Services,Housing & Living Conditions,Population Growth & Sustainability,Other,dominant_cluster
0,abdullah ii,Abdullah II,1,0.000166,0.000099,0.000033,0.000033,0.000066,0.000033,0.000033,0.000033,0.000066,0.999437,Other
1,abir al sahlani,Abir Al-Sahlani,2,0.010289,0.142624,0.000082,0.000082,0.031162,0.000082,0.000082,0.000082,0.000164,0.815351,Other
2,adam gierek,Adam Gierek,1,0.001136,0.000682,0.736202,0.000227,0.000455,0.000227,0.000227,0.000227,0.000455,0.260161,Migration & Population Change
3,adam kosa,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,3,0.030480,0.348529,0.043290,0.000285,0.000570,0.000285,0.000285,0.000285,0.041282,0.534709,Other
4,adam szejnfeld,Adam Szejnfeld,1,0.000813,0.000488,0.000163,0.000163,0.000325,0.000163,0.000163,0.000163,0.992195,0.005366,Population Growth & Sustainability
5,adrian vazquez lazara,AdriÃƒÂ¡n VÃƒÂ¡zquez LÃƒÂ¡zara,1,0.000877,0.000526,0.000175,0.000175,0.000351,0.000175,0.000175,0.000175,0.000351,0.997018,Other
6,adriana maldonado lopez,Adriana Maldonado LÃƒÂ³pez,2,0.993999,0.000400,0.000133,0.000133,0.000267,0.000133,0.000133,0.000133,0.000267,0.004401,Ageing & Pensions
7,agnes jongerius,Agnes Jongerius,2,0.001235,0.000741,0.000247,0.060907,0.000494,0.000247,0.049211,0.000247,0.000494,0.886177,Other
8,agnes le brun,AgnÃƒÂ¨s Le Brun,1,0.034527,0.391551,0.000215,0.000215,0.000430,0.000215,0.000215,0.000215,0.000430,0.571987,Other
9,agnieszka koz owska rajewicz,Agnieszka KozÃ…â€šowska-Rajewicz,1,0.000199,0.102772,0.000040,0.000040,0.000080,0.000040,0.000040,0.000040,0.065122,0.831629,Other


In [ ]:
# Purpose: Export processed outputs for downstream use.
# Notes: Documentation-only update; logic and execution order are unchanged.

# Save LDA cluster summary to the shared tables folder
csv_path = TABLES_DIR / "mep_cluster_summary.csv"
excel_path = TABLES_DIR / "mep_cluster_summary.xlsx"

mep_cluster_summary.to_csv(csv_path, index=False, encoding="utf-8-sig")
mep_cluster_summary.to_excel(excel_path, index=False)

print("Saved CSV:", csv_path)
print("Saved Excel:", excel_path)


Saved CSV:  C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\6 LDA Topic Analysis\mep_cluster_summary.csv
Saved Excel: C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\6 LDA Topic Analysis\mep_cluster_summary.xlsx


# LLM Polarisation Score

This notebook computes an LLM-based polarisation score for MEPs.

## Data Sources
- `mep_cluster_summary.xlsx` Ã¢â‚¬â€ MEP-level average topic cluster shares from the LDA analysis.
- `mep_wikipedia_final.xlsx` Ã¢â‚¬â€ MEP biographical metadata scraped from Wikipedia.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ------------------------------
# Load LDA summary and Wikipedia metadata
# ------------------------------
mep_cluster_df = pd.read_excel(CLUSTER_SUMMARY_PATH)
mep_wiki_df = pd.read_excel(WIKIPEDIA_PATH)

print("mep_cluster_summary shape:", mep_cluster_df.shape)
print("mep_wikipedia_final shape:", mep_wiki_df.shape)

display(mep_cluster_df.head())
display(mep_wiki_df.head())


mep_cluster_summary shape: (668, 14)
mep_wikipedia_final shape:  (1918, 12)


,mep_name_clean,mep_name_raw,n_documents,Ageing & Pensions,Fertility & Family Formation,Migration & Population Change,Youth & Future Generations,Labour Force & Workforce,Regional Depopulation,Health & Care Services,Housing & Living Conditions,Population Growth & Sustainability,Other,dominant_cluster
0,abdullah ii,Abdullah II,1,0.000166,0.000099,0.000033,0.000033,0.000066,0.000033,0.000033,0.000033,0.000066,0.999437,Other
1,abir al sahlani,Abir Al-Sahlani,2,0.010289,0.142624,0.000082,0.000082,0.031162,0.000082,0.000082,0.000082,0.000164,0.815351,Other
2,adam gierek,Adam Gierek,1,0.001136,0.000682,0.736202,0.000227,0.000455,0.000227,0.000227,0.000227,0.000455,0.260161,Migration & Population Change
3,adam kosa,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,3,0.030480,0.348529,0.043290,0.000285,0.000570,0.000285,0.000285,0.000285,0.041282,0.534709,Other
4,adam szejnfeld,Adam Szejnfeld,1,0.000813,0.000488,0.000163,0.000163,0.000325,0.000163,0.000163,0.000163,0.992195,0.005366,Population Growth & Sustainability


,name,wikipedia_language,page_title,page_url,wikipedia_content_html,wikipedia_content_wikitext,page_title_en,page_url_en,wikipedia_content_html_en,wikipedia_content_wikitext_en,Unnamed: 10,Country
0,BARBAGALLO Giovanni,it,Giovanni Barbagallo,https://it.wikipedia.org/wiki/Giovanni_Barbagallo,"<div class=""mw-content-ltr mw-parser-output"" l...",{{disambiguazione}}\n\n* '''[[Giovanni Barbaga...,NaN,NaN,NaN,NaN,NaN,IT
1,CRUZ Joaquim,pt,Joaquim Cruz,https://pt.wikipedia.org/wiki/Joaquim_Cruz,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Info/esporte/atleta\n|olimpico = sim\n|nome ...,Joaquim Cruz,https://en.wikipedia.org/wiki/Joaquim_Cruz,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|Brazilian middle-distance ...,NaN,PT
2,DIETRICH Roberto,ro,Dietrich BrÃƒÂ¼ggemann,https://ro.wikipedia.org/wiki/Dietrich_Br%C3%B...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Infocaseta Actor}}\n'''Dietrich BrÃƒÂ¼ggemann''...,Dietrich BrÃƒÂ¼ggemann,https://en.wikipedia.org/wiki/Dietrich_Br%C3%B...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|German film director}}\n{{...,NaN,RO
3,GKIKOPOULOS Georgios,el,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GR
4,HIDVÃƒâ€°GHI BalÃƒÂ¡zs,hu,HidvÃƒÂ©ghi BalÃƒÂ¡zs,https://hu.wikipedia.org/wiki/Hidv%C3%A9ghi_Ba...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{SzemÃƒÂ©ly infobox\n| kÃƒÂ©p = HidvÃƒÂ©ghi BalÃƒÂ¡zs.jpg...,BalÃƒÂ¡zs HidvÃƒÂ©ghi,https://en.wikipedia.org/wiki/Bal%C3%A1zs_Hidv...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|Hungarian politician}}\n{{...,NaN,HU


## Fuzzy Name Matching

MEP names may differ slightly between the LDA cluster table and the Wikipedia table
(e.g. accents, name order, middle initials).

The approach:
1. Normalize names in both tables (lowercase, strip accents, collapse whitespace).
2. Use `rapidfuzz` for fast token-sort ratio matching to find the best Wikipedia candidate for each cluster-table MEP.
3. Keep matches above a configurable similarity threshold.
4. Merge the two tables on the matched names.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [55]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# pip install rapidfuzz  (fast fuzzy string matching)
try:
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "rapidfuzz"], check=True)


# ------------------------------
# Name normalization
# ------------------------------
def normalize_name(name) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    # Strip accents
    name = "".join(
        c for c in unicodedata.normalize("NFKD", name)
        if not unicodedata.combining(c)
    )
    # Lowercase, keep only letters/digits/spaces
    name = name.lower()
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    return re.sub(r"\s+", " ", name).strip()

# Detect the name column in each DataFrame
def detect_name_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

NAME_CANDIDATES = ["mep_name_raw", "mep_name_clean", "name", "full_name", "mep"]

cluster_name_col = detect_name_col(mep_cluster_df, NAME_CANDIDATES)
wiki_name_col    = detect_name_col(mep_wiki_df,    NAME_CANDIDATES)

if not cluster_name_col:
    raise ValueError(f"No name column found in mep_cluster_df. Columns: {list(mep_cluster_df.columns)}")
if not wiki_name_col:
    raise ValueError(f"No name column found in mep_wiki_df. Columns: {list(mep_wiki_df.columns)}")

print(f"Matching on: cluster['{cluster_name_col}'] <-> wiki['{wiki_name_col}']")

# Build normalized name lists
cluster_names_norm = mep_cluster_df[cluster_name_col].apply(normalize_name).tolist()
wiki_names_norm    = mep_wiki_df[wiki_name_col].apply(normalize_name).tolist()

# ------------------------------
# Fuzzy match: for each cluster MEP find best wiki match
# Uses token_sort_ratio so name-order differences don't matter
# ------------------------------
MATCH_THRESHOLD = 80  # 0-100; lower = more lenient

match_records = []
for i, cn in enumerate(cluster_names_norm):
    if not cn:
        match_records.append({"cluster_idx": i, "wiki_idx": None, "match_score": 0, "wiki_name_matched": None})
        continue

    result = process.extractOne(
        cn,
        wiki_names_norm,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=MATCH_THRESHOLD,
    )

    if result:
        matched_str, score, wiki_idx = result
        match_records.append({"cluster_idx": i, "wiki_idx": wiki_idx, "match_score": score, "wiki_name_matched": wiki_names_norm[wiki_idx]})
    else:
        match_records.append({"cluster_idx": i, "wiki_idx": None, "match_score": 0, "wiki_name_matched": None})

match_df = pd.DataFrame(match_records)

# How many matched?
n_matched  = match_df["wiki_idx"].notna().sum()
n_total    = len(match_df)
print(f"Matched {n_matched} / {n_total} MEPs (threshold={MATCH_THRESHOLD})")

# ------------------------------
# Build merged DataFrame
# ------------------------------
cluster_indexed = mep_cluster_df.reset_index(drop=True).copy()
wiki_indexed    = mep_wiki_df.reset_index(drop=True).copy()

# Suffix _wiki to avoid column name collisions
wiki_indexed.columns = [
    c if c == wiki_name_col else f"{c}_wiki"
    for c in wiki_indexed.columns
]

# Pull wiki rows for matched indices
wiki_rows = match_df["wiki_idx"].map(
    lambda idx: wiki_indexed.iloc[int(idx)].to_dict() if pd.notna(idx) else {}
)
wiki_part = pd.DataFrame(wiki_rows.tolist(), index=cluster_indexed.index)

mep_merged = pd.concat([cluster_indexed, wiki_part], axis=1)
mep_merged["match_score"] = match_df["match_score"].values
mep_merged["wiki_name_matched"] = match_df["wiki_name_matched"].values

print("Merged table shape:", mep_merged.shape)
mep_merged.head()


Matching on: cluster['mep_name_raw'] <-> wiki['name']
Matched 427 / 668 MEPs (threshold=80)
Merged table shape: (668, 28)


,mep_name_clean,mep_name_raw,n_documents,Ageing & Pensions,Fertility & Family Formation,Migration & Population Change,Youth & Future Generations,Labour Force & Workforce,Regional Depopulation,Health & Care Services,...,wikipedia_content_html_wiki,wikipedia_content_wikitext_wiki,page_title_en_wiki,page_url_en_wiki,wikipedia_content_html_en_wiki,wikipedia_content_wikitext_en_wiki,Unnamed: 10_wiki,Country_wiki,match_score,wiki_name_matched
0,abdullah ii,Abdullah II,1,0.000166,0.000099,0.000033,0.000033,0.000066,0.000033,0.000033,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None
1,abir al sahlani,Abir Al-Sahlani,2,0.010289,0.142624,0.000082,0.000082,0.031162,0.000082,0.000082,...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Infobox politiker\n|namn = Abi...,Abir Al-Sahlani,https://en.wikipedia.org/wiki/Abir_Al-Sahlani,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Short description|Swedish politician (born 1...,NaN,NaN,100.0,al sahlani abir
2,adam gierek,Adam Gierek,1,0.001136,0.000682,0.736202,0.000227,0.000455,0.000227,0.000227,...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Naukowiec infobox\n |imiÃ„â„¢ i nazwisko ...,Adam Gierek,https://en.wikipedia.org/wiki/Adam_Gierek,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Short description|Polish politician}}\n{{Use...,NaN,PL,100.0,gierek adam
3,adam kosa,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,3,0.030480,0.348529,0.043290,0.000285,0.000570,0.000285,0.000285,...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{SzemÃƒÂ©ly infobox\n| nÃƒÂ©v = dr. KÃƒÂ³sa ÃƒÂdÃƒÂ¡m\n| tÃƒÂ­...,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,https://en.wikipedia.org/wiki/%C3%81d%C3%A1m_K...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Short description|Hungarian politician}}\n{{...,NaN,HU,100.0,kosa adam
4,adam szejnfeld,Adam Szejnfeld,1,0.000813,0.000488,0.000163,0.000163,0.000325,0.000163,0.000163,...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Biogram infobox\n |imiÃ„â„¢ i nazwisko ...,Adam Szejnfeld,https://en.wikipedia.org/wiki/Adam_Szejnfeld,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Short description|Polish politician (born 19...,NaN,PL,100.0,szejnfeld adam


## Relevant Speech Corpus For Polarisation Scoring

This section prepares the speech corpus used for LLM scoring.

Updated logic:
1. Keep all matched MEPs in the scoring base (do not drop MEPs by dominant cluster).
2. Use LDA-classified documents from `docs_demography_lda`.
3. Keep only documents whose dominant topic maps to a cluster other than `Other`.
4. Aggregate those non-`Other` documents per MEP into one scoring corpus.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# Reuse canonical_name() defined earlier in the notebook.

# ------------------------------------------------------------------
# 1) Keep all matched MEPs in the scoring base (no dominant-cluster exclusion).
# ------------------------------------------------------------------
mep_scoring_base = mep_merged.copy()
mep_scoring_base = mep_scoring_base[mep_scoring_base["match_score"].fillna(0) > 0].copy()

# ------------------------------------------------------------------
# 2) Build topic -> cluster mapping to identify which documents are 'Other'.
# ------------------------------------------------------------------
if "topic_clustered_df" in globals() and topic_clustered_df is not None and not topic_clustered_df.empty:
    topic_to_cluster = dict(zip(topic_clustered_df["topic"], topic_clustered_df["cluster"]))
else:
    topic_to_cluster = {}
    for cluster_name, indices in CLUSTER_MAP.items():
        for idx in indices:
            topic_to_cluster[f"topic_{idx:02d}"] = cluster_name

# ------------------------------------------------------------------
# 3) Use LDA-classified documents and keep only non-'Other' dominant cluster docs.
# ------------------------------------------------------------------
required_doc_cols = ["mep_name_clean", "mep_name_raw", "text", "dominant_topic"]
missing_doc_cols = [c for c in required_doc_cols if c not in docs_demography_lda.columns]
if missing_doc_cols:
    raise ValueError(f"docs_demography_lda is missing required columns: {missing_doc_cols}")

docs_for_scoring = docs_demography_lda[required_doc_cols].copy()
docs_for_scoring["dominant_cluster_doc"] = docs_for_scoring["dominant_topic"].map(topic_to_cluster).fillna("Other")
docs_for_scoring = docs_for_scoring[docs_for_scoring["dominant_cluster_doc"] != "Other"].copy()

# ------------------------------------------------------------------
# 4) Aggregate non-'Other' documents per MEP for LLM scoring input.
# ------------------------------------------------------------------
mep_relevant_speeches = (
    docs_for_scoring
    .dropna(subset=["mep_name_clean", "text"] )
    .groupby(["mep_name_clean", "mep_name_raw"], as_index=False)
    .agg(
        relevant_speech_text=("text", lambda x: "\n\n".join(x.dropna().astype(str))),
        n_relevant_speeches=("text", "size"),
    )
)

mep_scoring_df = mep_scoring_base.merge(
    mep_relevant_speeches,
    on=["mep_name_clean", "mep_name_raw"],
    how="left",
)

mep_scoring_df = mep_scoring_df[mep_scoring_df["relevant_speech_text"].fillna("").str.strip() != ""].reset_index(drop=True)

print("Matched MEPs (all) in base:", mep_scoring_base.shape)
print("MEPs with non-'Other' speech docs for scoring:", mep_scoring_df.shape)
mep_scoring_df[["mep_name_raw", "dominant_cluster", "n_relevant_speeches"]].head()


Matched MEPs (all) in base: (427, 28)
MEPs with non-'Other' speech docs for scoring: (161, 30)


,mep_name_raw,dominant_cluster,n_relevant_speeches
0,Adam Gierek,Migration & Population Change,1.0
1,ÃƒÂdÃƒÂ¡m KÃƒÂ³sa,Other,1.0
2,Adam Szejnfeld,Population Growth & Sustainability,1.0
3,AgnÃƒÂ¨s Le Brun,Other,1.0
4,Agnieszka KozÃ…â€šowska-Rajewicz,Ageing & Pensions,1.0


## LLM Polarisation Scoring

This section scores each matched MEP on the polarisation of their relevant speech corpus.

Design choices:
1. Score one aggregated relevant corpus per MEP.
2. Use free OpenRouter models with fallback across currently live `:free` endpoints.
3. Return structured JSON with a score, confidence, evidence, and dimension-level sub-scores.
4. Do not run this cell unless you want to trigger API requests.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Run the core processing logic for this stage of the notebook workflow.
# Notes: Documentation-only update; logic and execution order are unchanged.

# Do not hardcode API keys in notebooks.
# Preferred: set os.environ['OPENROUTER_API_KEY'] in the active kernel.
API_KEY = (os.environ.get('OPENROUTER_API_KEY') or '').strip()

if not API_KEY:
    print("OPENROUTER_API_KEY not found in environment. Set it before running scoring.")


In [58]:
# Purpose: Run the core processing logic for this stage of the notebook workflow.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ------------------------------
# OpenRouter API key pre-check
# ------------------------------
# 1) Preferred: set key in this kernel (remove angle brackets):
# os.environ['OPENROUTER_API_KEY'] = '<your_openrouter_key>'
# 2) Also accepted: define API_KEY = '<your_openrouter_key>' in a previous cell

preset_key = globals().get('API_KEY', '')
if not isinstance(preset_key, str):
    preset_key = ''

key = (
    preset_key.strip().strip('"').strip("'")
    or (os.environ.get('OPENROUTER_API_KEY') or '').strip()
    or (os.environ.get('OPENROUTER_KEY') or '').strip()
)

print('Key present in kernel:', bool(key))
if key:
    print('Key format looks OpenRouter-like:', key.startswith('sk-or-v1-'))
    try:
        resp = requests.get(
            'https://openrouter.ai/api/v1/models',
            headers={'Authorization': f'Bearer {key}'},
            timeout=30,
        )
        print('Auth test status:', resp.status_code)
        if resp.status_code == 401:
            print('Unauthorized: key is invalid/revoked, or not from OpenRouter.')
        elif resp.ok:
            print('Auth OK. You can run the scoring cell.')
        else:
            print('Auth failed with response:', resp.text[:300])
    except Exception as e:
        print('Auth request failed:', e)
else:
    print("No key found. Set OPENROUTER_API_KEY or API_KEY in this notebook, then rerun this cell.")


Key present in kernel: True
Key format looks OpenRouter-like: True
Auth test status: 200
Auth OK. You can run the scoring cell.


In [59]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

BASE_URL = 'https://openrouter.ai/api/v1'

# Read key from previous API_KEY variable (if set) or environment variables.
_preset_key = globals().get('API_KEY', '')
if not isinstance(_preset_key, str):
    _preset_key = ''

API_KEY = (
    _preset_key.strip().strip('"').strip("'")
    or (os.environ.get('OPENROUTER_API_KEY') or '').strip()
    or (os.environ.get('OPENROUTER_KEY') or '').strip()
)

if not API_KEY:
    raise ValueError(
        "Missing OpenRouter API key in this notebook kernel. "
        "Set API_KEY in a previous cell, or set os.environ['OPENROUTER_API_KEY'] and rerun."
    )

if not API_KEY.startswith('sk-or-v1-'):
    print('Warning: API key does not start with sk-or-v1-. Verify you are using an OpenRouter key.')

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Quick auth smoke-test for clearer errors before the scoring loop.
try:
    auth_check = requests.get(
        f'{BASE_URL}/models',
        headers={'Authorization': f'Bearer {API_KEY}'},
        timeout=30,
    )
    if auth_check.status_code == 401:
        raise ValueError(
            'OpenRouter authentication failed (401 Unauthorized). '
            'The key is invalid, expired, or not available in this kernel session.'
        )
    auth_check.raise_for_status()
except Exception as e:
    raise RuntimeError(
        f'OpenRouter auth check failed before scoring loop: {e}'
    )

STATIC_MODEL_CANDIDATES = [
    'meta-llama/llama-3.3-70b-instruct:free',
    'openai/gpt-oss-20b:free',
    'google/gemma-4-31b-it:free',
    'meta-llama/llama-3.2-3b-instruct:free'
]
MAX_INPUT_CHARS = 18000
MAX_OUTPUT_TOKENS = 1000
MAX_RETRIES_PER_MODEL = 2
BACKOFF_SECONDS = 1.5
RUN_POLARISATION_SCORING = True

def discover_live_free_models(limit: int = 25):
    try:
        resp = requests.get(f'{BASE_URL}/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=30)
        resp.raise_for_status()
        data = resp.json().get('data', [])
        ids = [m.get('id', '') for m in data if isinstance(m, dict)]
        return [m for m in ids if m.endswith(':free')][:limit]
    except Exception:
        return []

def build_model_candidates():
    merged = []
    for model_name in STATIC_MODEL_CANDIDATES + discover_live_free_models():
        model_name = (model_name or '').strip()
        if model_name and model_name not in merged:
            merged.append(model_name)
    return merged

MODEL_CANDIDATES = build_model_candidates()

SYSTEM = (
    'You assess political speech polarisation. Return only valid JSON. '
    'Score based on divisive rhetoric, us-versus-them framing, hostility toward opponents, '
    'moral absolutism, and emotionally escalatory language. '
    'Do not infer beyond the text. Provide short evidence quotes.'
)

def build_prompt(row):
    cluster_bits = []
    for col in row.index:
        if col in ['Ageing & Pensions', 'Fertility & Family Formation', 'Migration & Population Change', 'Youth & Future Generations', 'Labour Force & Workforce', 'Regional Depopulation', 'Health & Care Services', 'Housing & Living Conditions', 'Population Growth & Sustainability', 'Other']:
            cluster_bits.append(f'{col}: {row[col]}')
    cluster_text = '; '.join(cluster_bits)
    speech_text = str(row.get('relevant_speech_text', '') or '')[:MAX_INPUT_CHARS]
    return f'''MEP: {row.get('mep_name_raw', '')}
Dominant cluster: {row.get('dominant_cluster', '')}
Cluster shares: {cluster_text}
Number of relevant speeches: {row.get('n_relevant_speeches', '')}

Task:
Return ONLY valid JSON with this schema:
{{
  "polarisation_score": number,
  "confidence": number,
  "dimensions": {{
    "us_vs_them": number,
    "emotional_intensity": number,
    "moral_absolutism": number,
    "hostility_to_opponents": number
  }},
  "summary": string,
  "evidence_quotes": [string]
}}

Scale: 0 to 10, where 0 = non-polarised/technical and 10 = extremely polarising.

Relevant speech corpus:
"""{speech_text}"""'''

def safe_json_loads(s: str):
    s = str(s).strip()
    s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s*```$', '', s)
    match = re.search(r'\{.*\}', s, flags=re.DOTALL)
    if match:
        s = match.group(0).strip()
    s = re.sub(r',\s*([}\]])', r'\1', s)
    return json.loads(s)

def call_free_model(prompt: str):
    errors = []
    for model_name in MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            try:
                resp = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {'role': 'system', 'content': SYSTEM},
                        {'role': 'user', 'content': prompt},
                    ],
                    temperature=0.0,
                    max_tokens=MAX_OUTPUT_TOKENS,
                )
                content = (resp.choices[0].message.content or '').strip()
                return model_name, safe_json_loads(content)
            except Exception as e:
                errors.append(f'{model_name} attempt {attempt}: {e}')
                if '404' in str(e) or 'unavailable for free' in str(e).lower() or 'no endpoints found' in str(e).lower():
                    break
                time.sleep(BACKOFF_SECONDS * attempt)
    raise RuntimeError('All free model attempts failed: ' + ' | '.join(errors[-8:]))

if not RUN_POLARISATION_SCORING:
    print('Set RUN_POLARISATION_SCORING = True to run the LLM scoring loop.')
else:
    polarisation_rows = []
    for _, row in mep_scoring_df.iterrows():
        model_name, parsed = call_free_model(build_prompt(row))
        polarisation_rows.append({
            'mep_name_clean': row.get('mep_name_clean'),
            'mep_name_raw': row.get('mep_name_raw'),
            'model_used': model_name,
            'n_relevant_speeches': row.get('n_relevant_speeches'),
            'dominant_cluster': row.get('dominant_cluster'),
            'polarisation_score': parsed.get('polarisation_score'),
            'confidence': parsed.get('confidence'),
            'us_vs_them': (parsed.get('dimensions') or {}).get('us_vs_them'),
            'emotional_intensity': (parsed.get('dimensions') or {}).get('emotional_intensity'),
            'moral_absolutism': (parsed.get('dimensions') or {}).get('moral_absolutism'),
            'hostility_to_opponents': (parsed.get('dimensions') or {}).get('hostility_to_opponents'),
            'summary': parsed.get('summary'),
            'evidence_quotes': json.dumps(parsed.get('evidence_quotes', []), ensure_ascii=False)
        })
    polarisation_scores_df = pd.DataFrame(polarisation_rows)
    print('Polarisation scores shape:', polarisation_scores_df.shape)
    polarisation_scores_df.head()


Polarisation scores shape: (161, 13)


## Export step

Run the next export cell only after polarisation scoring has produced polarisation_scores_df.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


## Export Polarisation Scores

Save `polarisation_scores_df` as CSV so it can be used in downstream regression analysis.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# Save polarisation outputs to the shared tables folder.
POLARISATION_CSV_PATH = TABLES_DIR / "polarisation_scores.csv"
POLARISATION_SCORED_ONLY_PATH = TABLES_DIR / "polarisation_scores_scored_only.csv"

if "polarisation_scores_df" not in globals() or polarisation_scores_df is None or polarisation_scores_df.empty:
    raise ValueError("polarisation_scores_df is missing or empty. Run the scoring cell first.")

# Build the all-MEP base table (preferred order: mep_cluster_df -> mep_cluster_summary -> mep_merged).
if "mep_cluster_df" in globals() and mep_cluster_df is not None and not mep_cluster_df.empty:
    all_meps_base = mep_cluster_df.copy()
elif "mep_cluster_summary" in globals() and mep_cluster_summary is not None and not mep_cluster_summary.empty:
    all_meps_base = mep_cluster_summary.copy()
elif "mep_merged" in globals() and mep_merged is not None and not mep_merged.empty:
    all_meps_base = mep_merged.copy()
else:
    raise ValueError("No all-MEP base dataframe found. Expected mep_cluster_df, mep_cluster_summary, or mep_merged.")

required_keys = ["mep_name_clean", "mep_name_raw"]
missing_keys = [k for k in required_keys if k not in all_meps_base.columns]
if missing_keys:
    raise ValueError(f"All-MEP base dataframe is missing key columns: {missing_keys}")

all_meps_base = all_meps_base.drop_duplicates(subset=required_keys).copy()

polarisation_scores_all_meps = all_meps_base.merge(
    polarisation_scores_df,
    on=required_keys,
    how="left",
    suffixes=("", "_score"),
)

def _norm_name(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# Enrich with original extracted-file MEP name if available.
if EXTRACTED_POLARISATION_PATH.exists():
    extracted_df = pd.read_excel(EXTRACTED_POLARISATION_PATH).copy()
    old_name_col = "mep_name_old_from_extracted" if "mep_name_old_from_extracted" in extracted_df.columns else None

    if old_name_col is not None:
        extracted_name_candidates = ["mep_name_raw", "name", "full_name_extracted", old_name_col]
        extracted_name_col = next((c for c in extracted_name_candidates if c in extracted_df.columns), None)

        if extracted_name_col is not None:
            extracted_df["_name_norm"] = extracted_df[extracted_name_col].fillna("").astype(str).map(_norm_name)
            old_name_map = (
                extracted_df[["_name_norm", old_name_col]]
                .dropna(subset=["_name_norm"] )
                .drop_duplicates(subset=["_name_norm"], keep="first")
            )

            polarisation_scores_all_meps["_name_norm"] = polarisation_scores_all_meps["mep_name_raw"].fillna("").astype(str).map(_norm_name)
            polarisation_scores_all_meps = polarisation_scores_all_meps.merge(old_name_map, on="_name_norm", how="left")
            polarisation_scores_all_meps = polarisation_scores_all_meps.drop(columns=["_name_norm"], errors="ignore")
        else:
            polarisation_scores_all_meps["mep_name_old_from_extracted"] = pd.NA
    else:
        polarisation_scores_all_meps["mep_name_old_from_extracted"] = pd.NA
else:
    polarisation_scores_all_meps["mep_name_old_from_extracted"] = pd.NA

try:
    polarisation_scores_all_meps.to_csv(POLARISATION_CSV_PATH, index=False, encoding="utf-8-sig")
    saved_all_path = POLARISATION_CSV_PATH
except PermissionError:
    saved_all_path = TABLES_DIR / "polarisation_scores_latest.csv"
    polarisation_scores_all_meps.to_csv(saved_all_path, index=False, encoding="utf-8-sig")

scored_only_export = polarisation_scores_all_meps[
    polarisation_scores_all_meps["polarisation_score"].notna()
] .copy()

try:
    scored_only_export.to_csv(POLARISATION_SCORED_ONLY_PATH, index=False, encoding="utf-8-sig")
    saved_scored_path = POLARISATION_SCORED_ONLY_PATH
except PermissionError:
    saved_scored_path = TABLES_DIR / "polarisation_scores_scored_only_latest.csv"
    scored_only_export.to_csv(saved_scored_path, index=False, encoding="utf-8-sig")

print("Saved ALL-MEP polarisation CSV:", saved_all_path)
print("All-MEP rows:", len(polarisation_scores_all_meps))
print("Saved scored-only CSV:", saved_scored_path)
print("Scored-only rows:", len(scored_only_export))


Saved ALL-MEP polarisation CSV: C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\6 LDA Topic Analysis\polarisation_scores.csv
All-MEP rows: 668
Saved scored-only CSV: C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\6 LDA Topic Analysis\polarisation_scores_scored_only.csv
Scored-only rows: 161
